# PA 766 | 1. Convert PDFs to text

**Goal:** turn the class PDFs into `.txt` files and check whether their text is readable.

Open this notebook in **Google Colab** using **File > Upload notebook**. Use a standard CPU runtime. Run the cells from top to bottom using the play buttons. No API key is needed.

Download the example PDFs supplied by your instructor from the course's **Data / PDF to Text** folder. Start with `D00000-ElectricVehicleScenarioAnalysisWorkshopSeries.pdf`; you can select all four PDFs when ready.

You will download one ZIP containing the text files. Keep it for Notebook 2.

## 1. Install and import the tools

PyMuPDF reads the existing text inside a PDF. These examples contain text layers. A scanned image needs optical character recognition (OCR) first; this notebook does not perform OCR.

In [3]:
%pip -q install pymupdf==1.28.2

In [4]:
from pathlib import Path  # Work with file and folder paths, such as locating PDFs and naming output files.
from zipfile import ZipFile  # Create or read ZIP archives to bundle multiple files.
import pymupdf  # Open PDFs and extract their text, images, and other content.
import pandas as pd  # Work with tables of data using the short name pd.
from google.colab import files  # Upload files to Colab or download files to your computer.
from IPython.display import display  # Show formatted output, such as tables, inside the notebook.

print("PyMuPDF version:", pymupdf.VersionBind)  # Display the installed PyMuPDF version.

PyMuPDF version: 1.28.2


## 2. Upload the PDFs

In [7]:
from google.colab import drive  # Access Google Drive from Colab.
from pathlib import Path  # Work with folder and file paths.

drive.mount("/content/drive")  # Connect your Google Drive; follow the authorization prompt.

Mounted at /content/drive


In [8]:
pdf_folder = Path("/content/drive/MyDrive/a-ncsu-courses/PA766-2026/Data/PDF to Text")  # Specify the folder containing your PDFs.

if not pdf_folder.is_dir():  # Check that the folder exists.
    raise FileNotFoundError(f"Folder not found: {pdf_folder}")  # Stop if the folder path is incorrect.



In [9]:
pdf_inputs = {  # Store PDF filenames and contents for the conversion code.
    path.name: path.read_bytes()  # Read each PDF's contents as bytes.
    for path in sorted(pdf_folder.iterdir())  # Examine files directly inside the folder in filename order.
    if path.is_file() and path.suffix.lower() == ".pdf"  # Keep only PDFs, ignoring capitalization.
}

if not pdf_inputs:  # Check whether any PDFs were found.
    raise ValueError(f"No PDFs found in: {pdf_folder}")  # Stop if the folder contains no PDFs.

print(f"Ready to convert {len(pdf_inputs)} PDF(s).")  # Display the number of PDFs ready for conversion.

Ready to convert 4 PDF(s).


## 3. Extract and save the text

For each PDF, we read one page at a time. We keep text blocks separated by blank lines and replace line breaks *within* a block with spaces. A PDF block is a layout region, so it does not always equal a paragraph.

The invisible page-break character `\f` separates pages in each `.txt` file. Notebook 2 uses it to recover the source page number. We retain headings, footers, and table text so you can inspect what the extractor produced.

The table flags pages with fewer than 80 extracted characters. A flagged page may be a cover, divider, chart, or scanned page. This is a prompt to inspect the PDF, not an automatic diagnosis. Other pages can also have extraction errors.

In [11]:
output_dir = Path("/content/drive/MyDrive/a-ncsu-courses/Temporary")  # Set the folder where converted text files will be saved.
output_dir.mkdir(exist_ok=True)  # Create the folder; allow it to already exist.
text_paths = []  # Store the paths of the text files we create.
summary = []  # Store conversion details for each PDF.

for filename, pdf_bytes in sorted(pdf_inputs.items()):  # Process uploaded PDFs in filename order.
    page_texts = []  # Store the extracted text for each page of this PDF.
    low_text_pages = []  # Track pages with little text that may need checking.
    with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:  # Open the PDF from its uploaded contents; close it afterward.
        for page_number, page in enumerate(pdf, start=1):  # Process each page, numbering from 1.
            blocks = page.get_text("blocks", sort=True)  # Extract blocks, sorted roughly from top to bottom and left to right.
            paragraphs = [" ".join(block[4].split())  # Clean each block's text by replacing repeated whitespace with single spaces.
                          for block in blocks if block[6] == 0 and block[4].strip()]  # Keep only nonempty text blocks.
            page_text = "\n\n".join(paragraphs)  # Separate the text blocks with blank lines.
            page_texts.append(page_text)  # Save this page's extracted text.
            if len(page_text.strip()) < 80:  # Check whether the page contains fewer than 80 characters of text.
                low_text_pages.append(page_number)  # Flag the page for review; it may be scanned or simply short.

    if not any(text.strip() for text in page_texts):  # Check whether every page is empty after extraction.
        raise ValueError(f"{filename}: no text found. Ask your instructor about OCR.")  # Stop; OCR can recognize text in scanned images.

    text_path = output_dir / (Path(filename).stem + ".txt")  # Build an output path using the PDF's name without its extension.
    text_path.write_text("\f".join(page_texts), encoding="utf-8")  # Save the text, separating pages with form-feed characters (\f).
    text_paths.append(text_path)  # Add the saved file's path to our list.
    summary.append({"source_pdf": filename, "pages": len(page_texts),  # Record the source filename and page count.
                    "words": sum(len(t.split()) for t in page_texts),  # Count whitespace-separated words across all pages.
                    "pages_to_check": ", ".join(map(str, low_text_pages)) or "None flagged"})  # List flagged page numbers, if any.

display(pd.DataFrame(summary))  # Display the conversion details as a table.

,source_pdf,pages,words,pages_to_check
0,D00000-ElectricVehicleScenarioAnalysisWorkshop...,15,5199,None flagged
1,D00001-TheEraOfFlatPowerDemandIsOver.pdf,29,6083,"2, 7, 13, 25"
2,D00002-CharacteristicsAndRiskOfEmergingLargeLo...,49,18336,None flagged
3,D00003-PlanningForAndManagingInterminateElecri...,20,5828,"6, 18"
